In [ ]:
import numpy as np
import pandas as pd

# Dữ liệu từ credit_card_default
attribute_names =  ['age', 'income','student', 'credit_rate']
class_name = 'default'
data1 ={
    'age' : ['youth', 'youth', 'middle_age', 'senior', 'senior', 'senior','middle_age', 'youth', 'youth', 'senior', 'youth', 'middle_age','middle_age', 'senior'],
    'income' : ['high', 'high', 'high', 'medium', 'low', 'low', 'low', 'medium','low', 'medium', 'medium', 'medium', 'high', 'medium'],
    'student' : ['no','no','no','no','yes','yes','yes','no','yes','yes','yes','no','yes','no'],
    'credit_rate' : ['fair', 'excellent', 'fair', 'fair', 'fair', 'excellent', 'excellent', 'fair', 'fair', 'fair','excellent', 'excellent', 'fair', 'excellent'],
    'default' : ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes','yes', 'yes', 'yes', 'no']
}
df1 = pd.DataFrame(data1, columns=data1.keys())
df1

,age,income,student,credit_rate,default
0,youth,high,no,fair,no
1,youth,high,no,excellent,no
2,middle_age,high,no,fair,yes
3,senior,medium,no,fair,yes
4,senior,low,yes,fair,yes
5,senior,low,yes,excellent,no
6,middle_age,low,yes,excellent,yes
7,youth,medium,no,fair,no
8,youth,low,yes,fair,yes
9,senior,medium,yes,fair,yes


In [ ]:
# BÀI TẬP 1: Tính Entropy của tập dữ liệu
# Gợi ý: Công thức Entropy = - sum(p * log2(p))
def entropy(value_counts):
    n = value_counts.sum()
    ent = 0
    for key in value_counts.keys():
        p = value_counts[key] / n
        if p > 0:
            ent -= p * np.log2(p)
    return ent

class_value_counts = df1[class_name].value_counts()
entropy_class = entropy(class_value_counts)
print(f'Entropy của tập dữ liệu gốc (Parent Entropy): {entropy_class:.3f}')

Entropy của tập dữ liệu gốc (Parent Entropy): 0.940


In [ ]:
# BÀI TẬP 2: Tính Conditional Entropy (Entropy có điều kiện) cho từng thuộc tính
def entropy_split_a(attribute_name):
    attribute_values = df1[attribute_name].value_counts()
    ent_A = 0
    for key in attribute_values.keys():
        attribute_values_counts = df1[df1[attribute_name] == key]
        df_k = attribute_values_counts[class_name].value_counts()
        n_k = attribute_values[key]
        n = df1.shape[0]
        ent_A += (n_k / n) * entropy(df_k)
    return ent_A

In [14]:
# BÀI TẬP 3: Tính Information Gain (Độ lợi thông tin) và chọn thuộc tính tốt nhất
ig_attribute = {}
# TODO: Duyệt qua danh sách attribute_names ['age', 'income','student', 'credit_rate']
# Tính Information Gain = Parent Entropy - Conditional Entropy
# Lưu kết quả vào dictionary ig_attribute
for key in attribute_names:
    ig_attribute[key] = entropy_class - entropy_split_a(key)
    print(f"Information Gain của {key:<12} là {ig_attribute[key]:.3f}")
best_attribute_ig = max(ig_attribute, key=ig_attribute.get)
print(f'\n=> Thuộc tính được chọn làm nút gốc (chỉ số cao nhất) là: {best_attribute_ig}')

Information Gain của age          là 0.247
Information Gain của income       là 0.029
Information Gain của student      là 0.152
Information Gain của credit_rate  là 0.048

=> Thuộc tính được chọn làm nút gốc (chỉ số cao nhất) là: age


In [15]:
# STEP 1: Calculate gini(D)
def gini_impurity (value_counts):
    n = value_counts.sum()
    p_sum = 0
    for key in value_counts.keys():
        p_sum = p_sum  +  (value_counts[key] / n ) * (value_counts[key] / n )
    gini = 1 - p_sum
    return gini

class_value_counts = df1[class_name].value_counts()
print(f'Number of samples in each class is:\n{class_value_counts}')

gini_class = gini_impurity(class_value_counts)
print(f'\nGini Impurity of the class is {gini_class:.3f}')

Number of samples in each class is:
default
yes    9
no     5
Name: count, dtype: int64

Gini Impurity of the class is 0.459


In [16]:
# STEP 2:
# Calculate the weighted Gini impurity for each candidate attribute.
def gini_split_a(attribute_name):
    attribute_values = df1[attribute_name].value_counts()
    gini_A = 0
    for key in attribute_values.keys():
        df_k = df1[class_name][df1[attribute_name] == key].value_counts()
        n_k = attribute_values[key]
        n = df1.shape[0]
        gini_A = gini_A + ((n_k / n) * gini_impurity(df_k))
    return gini_A

gini_attribute = {}
for key in attribute_names:
    gini_attribute[key] = gini_split_a(key)
    print(f'Gini for {key} is {gini_attribute[key]:.3f}')


Gini for age is 0.343
Gini for income is 0.440
Gini for student is 0.367
Gini for credit_rate is 0.429


In [17]:
# STEP 3:
# Select the attribute with the minimum weighted Gini impurity.
# Gini decrease = Gini(parent) - weighted Gini(children).

min_split_gini = min(gini_attribute.values())
max_gini_decrease = gini_class - min_split_gini

print(f'Parent Gini                  : {gini_class:.3f}')
print(f'Minimum weighted Gini        : {min_split_gini:.3f}')
print(f'Maximum Gini decrease        : {max_gini_decrease:.3f}')

selected_attribute = min(gini_attribute, key=gini_attribute.get)
print('The selected attribute is    :', selected_attribute)


Parent Gini                  : 0.459
Minimum weighted Gini        : 0.343
Maximum Gini decrease        : 0.116
The selected attribute is    : age
